# AurisCore spectrogram-CNN measured training report

This notebook runs the **complete real-data pipeline by default** using `configs/heart_spectrogram_cnn.yaml`: manifest construction, participant splitting, preprocessing, spectrogram-CNN training, validation-only threshold selection, measurement, plots, and report export.

It does not evaluate the final test partition. This is a research screening experiment, not a clinical validation. Full training overwrites the current development CNN artifacts.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import hashlib
import json
import os
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    auc, confusion_matrix, precision_recall_curve, roc_curve
)

cwd = Path.cwd().resolve()
if (cwd / 'src' / 'auriscore').exists():
    ROOT = cwd
elif (cwd.parent / 'src' / 'auriscore').exists():
    ROOT = cwd.parent
else:
    raise RuntimeError('Open this notebook from AI-Pipeline/ or AI-Pipeline/notebooks/.')
sys.path.insert(0, str(ROOT / 'src'))

from auriscore.cnn import _window, build_cnn_model, prepare_spectrogram_config
from auriscore.config import load_config
from auriscore.cross_validation import cross_validate_cnn
from auriscore.io import read_table
from auriscore.pipeline import run, stamp
from auriscore.splitting import assert_no_leakage

REPORT_DIR = ROOT / 'artifacts' / 'notebook_report'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Pipeline root: {ROOT}')
print(f'Report output: {REPORT_DIR}')

Pipeline root: C:\Users\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline
Report output: C:\Users\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline\artifacts\notebook_report


## Run controls

`RUN_FULL_TRAINING` is intentionally `True`. The run may take a long time and replaces the development model/metrics. Five-fold cross-validation is separate and disabled by default because it performs five more training runs.

In [6]:
CONFIG_PATH = ROOT / 'configs' / 'heart_spectrogram_cnn.yaml'
RUN_FULL_TRAINING = True
RUN_UNIT_TESTS = True
RUN_GROUPED_CROSS_VALIDATION = False
INSPECTION_WINDOWS_PER_CLASS = 12
SPECTROGRAM_PREVIEW_COUNT = 4

config = load_config(CONFIG_PATH)
assert config['model_type'] == 'cnn'
print(json.dumps({
    'config': str(CONFIG_PATH.relative_to(ROOT)),
    'run_full_training': RUN_FULL_TRAINING,
    'run_unit_tests': RUN_UNIT_TESTS,
    'run_grouped_cross_validation': RUN_GROUPED_CROSS_VALIDATION,
    'sample_rate': config['sample_rate'],
    'feature_fmax': config['feature_fmax'],
    'spectrogram_type': config.get('spectrogram_type'),
    'cnn_architecture': config.get('cnn_architecture'),
}, indent=2))

{
  "config": "configs\\heart_spectrogram_cnn.yaml",
  "run_full_training": true,
  "run_unit_tests": true,
  "run_grouped_cross_validation": false,
  "sample_rate": 8000,
  "feature_fmax": 2000,
  "spectrogram_type": "logmel",
  "cnn_architecture": "residual_se"
}


## Environment and automated tests

In [7]:
import tensorflow as tf

environment = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'processor': platform.processor(),
    'logical_cpu_count': os.cpu_count(),
    'tensorflow': tf.__version__,
    'tensorflow_devices': [device.name for device in tf.config.list_physical_devices()],
}
display(pd.Series(environment, name='value').to_frame())

test_result = {'executed': False}
if RUN_UNIT_TESTS:
    started = perf_counter()
    completed = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q'],
        cwd=ROOT, capture_output=True, text=True, check=False
    )
    test_result = {
        'executed': True,
        'return_code': completed.returncode,
        'elapsed_seconds': perf_counter() - started,
        'summary': next(
            (line for line in reversed(completed.stdout.splitlines()) if 'passed' in line),
            completed.stdout.strip().splitlines()[-1] if completed.stdout.strip() else ''
        ),
    }
    print(completed.stdout[-3000:])
    if completed.returncode != 0:
        print(completed.stderr[-3000:])
        raise RuntimeError('Automated tests failed; full training was not started.')
display(pd.Series(test_result, name='value').to_frame())

,value
generated_at_utc,2026-09-26T05:44:08.998247+00:00
python,3.13.7
platform,Windows-11-10.0.26100-SP0
processor,"AMD64 Family 25 Model 117 Stepping 2, Authenti..."
logical_cpu_count,16
tensorflow,2.21.0
tensorflow_devices,[/physical_device:CPU:0]


ers\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline\.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85
  c:\Users\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline\.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
    parse = parser.parseString(pattern)

.venv\Lib\site-packages\matplotlib\_fontconfig_pattern.py:8

,value
executed,True
return_code,0
elapsed_seconds,16.746581
summary,"[33m[32m34 passed[0m, [33m[1m76 warnings..."


## Complete training run

This is the long-running cell. It rebuilds derived data with the exact new configuration before training, so provenance checks stay valid.

In [8]:
training_run = {'executed': False}
if RUN_FULL_TRAINING:
    started = perf_counter()
    training_output = run(ROOT, config, stage='run_pipeline', download=False, synthetic=False)
    training_run = {
        'executed': True,
        'elapsed_seconds': perf_counter() - started,
        'status': training_output.get('status'),
        'training_epochs': training_output.get('training_epochs'),
        'best_validation_loss': training_output.get('best_validation_loss'),
    }
display(pd.Series(training_run, name='value').to_frame())

Epoch 1/50
665/665 ━━━━━━━━━━━━━━━━━━━━ 386s 549ms/step - loss: 0.6525 - pr_auc: 0.3943 - precision: 0.3388 - roc_auc: 0.6308 - sensitivity: 0.4840 - val_loss: 0.7622 - val_pr_auc: 0.5541 - val_precision: 0.2307 - val_roc_auc: 0.7670 - val_sensitivity: 0.9268
Epoch 2/50
665/665 ━━━━━━━━━━━━━━━━━━━━ 401s 578ms/step - loss: 0.5870 - pr_auc: 0.5493 - precision: 0.4517 - roc_auc: 0.7210 - sensitivity: 0.5228 - val_loss: 0.5755 - val_pr_auc: 0.6319 - val_precision: 0.3370 - val_roc_auc: 0.8023 - val_sensitivity: 0.8236
Epoch 3/50
665/665 ━━━━━━━━━━━━━━━━━━━━ 398s 575ms/step - loss: 0.5476 - pr_auc: 0.6148 - precision: 0.5135 - roc_auc: 0.7628 - sensitivity: 0.5563 - val_loss: 0.7982 - val_pr_auc: 0.6706 - val_precision: 0.8310 - val_roc_auc: 0.7980 - val_sensitivity: 0.4484
Epoch 4/50
665/665 ━━━━━━━━━━━━━━━━━━━━ 382s 551ms/step - loss: 0.5170 - pr_auc: 0.6520 - precision: 0.5582 - roc_auc: 0.7783 - sensitivity: 0.5701 - val_loss: 0.5016 - val_pr_auc: 0.7279 - val_precision: 0.4868 - val_ro

,value
executed,True
elapsed_seconds,7520.113338
status,real_data_development_evaluation
training_epochs,19
best_validation_loss,0.432462


## Dataset measurements and leakage check

In [9]:
segments_path = ROOT / 'data' / 'processed' / 'segments.csv'
if not segments_path.exists():
    raise FileNotFoundError('No segments.csv. Run the full-training cell first.')
segments = read_table(segments_path)
assert_no_leakage(segments)

dataset_rows = []
dataset_measurements = {}
for split_name, part in segments[segments.split.isin(['train', 'validation', 'test'])].groupby('split'):
    subjects = part[['subject_group', 'label']].drop_duplicates()
    row = {
        'split': split_name,
        'participants': int(part.subject_group.nunique()),
        'recordings': int(part.recording_id.nunique()),
        'windows': int(len(part)),
        'absent_participants': int((subjects.label == 'Absent').sum()),
        'present_participants': int((subjects.label == 'Present').sum()),
    }
    dataset_rows.append(row)
    dataset_measurements[split_name] = row
dataset_table = pd.DataFrame(dataset_rows).set_index('split')
display(dataset_table)

ax = dataset_table[['absent_participants', 'present_participants']].plot.bar(
    figsize=(8, 4), color=['#3b82f6', '#ef4444']
)
ax.set(title='Participant class distribution by split', ylabel='Participants', xlabel='Split')
ax.legend(['Absent', 'Present'])
plt.xticks(rotation=0)
plt.tight_layout()
class_plot = FIGURE_DIR / 'participant_class_distribution.png'
plt.savefig(class_plot, dpi=150)
plt.show()

,participants,recordings,windows,absent_participants,present_participants
split,,,,,
test,122,458,3530,99,23
train,568,2070,15954,458,110
validation,122,448,3335,98,24


C:\Users\Andi Alvin\AppData\Local\Temp\ipykernel_13436\1729914651.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Measured spectrogram tensors and previews

Normalization statistics below are fitted only from sampled training windows for this inspection. The actual training cell fits them from every training window and saves them with the model.

In [10]:
training_segments = segments[segments.split.eq('train')]
inspection_parts = []
for label, part in training_segments.groupby('label'):
    inspection_parts.append(part.sample(
        n=min(INSPECTION_WINDOWS_PER_CLASS, len(part)), random_state=int(config['seed'])
    ))
inspection_segments = pd.concat(inspection_parts).sort_values(['label', 'segment_id'])
inspection_config = prepare_spectrogram_config(
    inspection_segments, ROOT, dict(config, spectrogram_cache_enabled=False)
)
loaded_audio = {}
def audio_loader(path):
    if path not in loaded_audio:
        loaded_audio[path] = np.load(path, allow_pickle=False)
    return loaded_audio[path]

started = perf_counter()
tensors = [
    _window(row, ROOT, inspection_config, audio_loader)[..., 0]
    for row in inspection_segments.to_dict('records')
]
feature_elapsed = perf_counter() - started
stacked = np.stack(tensors)
spectrogram_measurements = {
    'sampled_windows': len(tensors),
    'tensor_shape': list(stacked.shape[1:]),
    'dtype': str(stacked.dtype),
    'minimum': float(stacked.min()),
    'maximum': float(stacked.max()),
    'mean': float(stacked.mean()),
    'standard_deviation': float(stacked.std()),
    'nonfinite_values': int((~np.isfinite(stacked)).sum()),
    'total_extraction_seconds': feature_elapsed,
    'milliseconds_per_window': 1000 * feature_elapsed / len(tensors),
    'frequency_statistics_fit_on_sampled_training_windows': len(inspection_segments),
}
display(pd.Series(spectrogram_measurements, name='value').to_frame())

preview_count = min(SPECTROGRAM_PREVIEW_COUNT, len(tensors))
fig, axes = plt.subplots(preview_count, 1, figsize=(10, 2.8 * preview_count), squeeze=False)
for axis, tensor, (_, row) in zip(axes[:, 0], tensors[:preview_count], inspection_segments.iloc[:preview_count].iterrows()):
    image = axis.imshow(tensor, origin='lower', aspect='auto', cmap='magma')
    axis.set(title=f"{row['label']} — {row['segment_id']}", xlabel='Time frame', ylabel='Frequency bin')
    fig.colorbar(image, ax=axis, fraction=0.02, pad=0.02)
fig.tight_layout()
spectrogram_plot = FIGURE_DIR / 'spectrogram_examples.png'
fig.savefig(spectrogram_plot, dpi=150)
plt.show()

,value
sampled_windows,24
tensor_shape,"[96, 313]"
dtype,float32
minimum,-4.689337
maximum,5.0
mean,-0.000005
standard_deviation,0.999976
nonfinite_values,0
total_extraction_seconds,0.060977
milliseconds_per_window,2.540725


C:\Users\Andi Alvin\AppData\Local\Temp\ipykernel_13436\669498757.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Architecture measurements

In [11]:
input_shape = (*stacked.shape[1:], 1)
inspection_model = build_cnn_model(input_shape, inspection_config)
architecture_measurements = {
    'model_name': inspection_model.name,
    'input_shape': list(input_shape),
    'trainable_parameters': int(sum(np.prod(weight.shape) for weight in inspection_model.trainable_weights)),
    'nontrainable_parameters': int(sum(np.prod(weight.shape) for weight in inspection_model.non_trainable_weights)),
    'total_parameters': int(inspection_model.count_params()),
}
display(pd.Series(architecture_measurements, name='value').to_frame())
inspection_model.summary()

,value
model_name,auriscore_heart_residual_se_cnn
input_shape,"[96, 313, 1]"
trainable_parameters,31151
nontrainable_parameters,1008
total_parameters,32159


Model: "auriscore_heart_residual_se_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spectrogram         │ (None, 96, 313,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 96, 313,   │        216 │ spectrogram[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 313,   │         96 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 96, 313,   │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_6  │ (None, 96, 313,   │        792 │ activation_7[0][… │
│ (SeparableConv2D)   │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 313,   │         96 │ separable_conv2d… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 96, 313,   │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_7  │ (None, 96, 313,   │        792 │ activation_8[0][… │
│ (SeparableConv2D)   │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 96, 313,   │         96 │ separable_conv2d… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 96, 313,   │          0 │ batch_normalizat… │
│                     │ 24)               │            │ activation_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 96, 313,   │          0 │ add_3[0][0]       │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1, 1, 24)  │          0 │ activation_9[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 1, 1, 4)   │        100 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 1, 1, 24)  │        120 │ conv2d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 96, 313,   │          0 │ activation_9[0][… │
│ (Multiply)          │ 24)               │            │ conv2d_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 48, 156,   │          0 │ multiply_3[0][0]  │
│ (MaxPooling2D)      │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_8  │ (None, 48, 156,   │      1,368 │ max_pooling2d_2[… │
│ (SeparableConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 32,159 (125.62 KB)

 Trainable params: 31,151 (121.68 KB)

 Non-trainable params: 1,008 (3.94 KB)

## Training curves

In [12]:
history_path = ROOT / 'artifacts' / 'metrics' / 'cnn_training_history.json'
if not history_path.exists():
    raise FileNotFoundError('Training history is missing. Complete the training cell first.')
history = json.loads(history_path.read_text(encoding='utf-8'))
epochs = np.arange(1, len(history['loss']) + 1)
curve_groups = [
    ('Loss', 'loss'), ('Sensitivity', 'sensitivity'), ('Precision', 'precision'),
    ('ROC-AUC', 'roc_auc'), ('PR-AUC', 'pr_auc'),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, (title, key) in zip(axes.flat, curve_groups):
    if key in history:
        axis.plot(epochs, history[key], label='Training')
    if f'val_{key}' in history:
        axis.plot(epochs, history[f'val_{key}'], label='Validation')
    axis.set(title=title, xlabel='Epoch', ylabel=title)
    axis.grid(alpha=0.25)
    axis.legend()
axes.flat[-1].axis('off')
fig.suptitle('Spectrogram CNN learning curves', y=1.02)
fig.tight_layout()
learning_plot = FIGURE_DIR / 'training_curves.png'
fig.savefig(learning_plot, dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Andi Alvin\AppData\Local\Temp\ipykernel_13436\1517813007.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Participant-level validation evaluation and threshold behavior

In [13]:
metrics_path = ROOT / 'artifacts' / 'metrics' / 'cnn_metrics.json'
predictions_path = ROOT / 'artifacts' / 'metrics' / 'cnn_validation_predictions.csv'
model_path = ROOT / 'artifacts' / 'models' / 'heart_cnn.keras'
metadata_path = ROOT / 'artifacts' / 'models' / 'heart_cnn.json'
metrics_payload = json.loads(metrics_path.read_text(encoding='utf-8'))
model_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
recording_predictions = pd.read_csv(predictions_path)
participant_predictions = recording_predictions.groupby('subject_group', as_index=False).agg(
    label=('label', 'first'), score=('score', 'mean')
)
participant_predictions['target'] = (participant_predictions.label == 'Present').astype(int)
threshold = float(metrics_payload['validation']['threshold'])
participant_predictions['prediction'] = (participant_predictions.score >= threshold).astype(int)
subject_metrics = metrics_payload['validation']['subject']
display(pd.Series(subject_metrics, name='value').drop('confusion_matrix').to_frame())

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for label, part in participant_predictions.groupby('label'):
    axes[0, 0].hist(part.score, bins=20, alpha=0.60, label=label)
axes[0, 0].axvline(threshold, color='black', linestyle='--', label=f'Threshold {threshold:.4f}')
axes[0, 0].set(title='Participant score distribution', xlabel='CNN score', ylabel='Participants')
axes[0, 0].legend()

fpr, tpr, _ = roc_curve(participant_predictions.target, participant_predictions.score)
axes[0, 1].plot(fpr, tpr, label=f"AUC {auc(fpr, tpr):.3f}")
axes[0, 1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0, 1].set(title='Participant ROC curve', xlabel='False-positive rate', ylabel='Sensitivity')
axes[0, 1].legend()

precision, recall, _ = precision_recall_curve(participant_predictions.target, participant_predictions.score)
axes[1, 0].plot(recall, precision, label=f"PR-AUC {auc(recall, precision):.3f}")
axes[1, 0].axhline(participant_predictions.target.mean(), linestyle='--', color='gray', label='Prevalence')
axes[1, 0].set(title='Participant precision-recall curve', xlabel='Recall', ylabel='Precision')
axes[1, 0].legend()

matrix = np.asarray(subject_metrics['confusion_matrix'])
image = axes[1, 1].imshow(matrix, cmap='Blues')
for (row, column), value in np.ndenumerate(matrix):
    axes[1, 1].text(column, row, str(value), ha='center', va='center')
axes[1, 1].set(
    title='Participant confusion matrix', xlabel='Predicted class', ylabel='True class',
    xticks=[0, 1], yticks=[0, 1], xticklabels=['Absent', 'Present'], yticklabels=['Absent', 'Present']
)
fig.colorbar(image, ax=axes[1, 1], fraction=0.046, pad=0.04)
fig.tight_layout()
evaluation_plot = FIGURE_DIR / 'participant_evaluation.png'
fig.savefig(evaluation_plot, dpi=150)
plt.show()

threshold_rows = []
targets = participant_predictions.target.to_numpy()
scores = participant_predictions.score.to_numpy()
for candidate in np.unique(scores):
    predicted = scores >= candidate
    tn, fp, fn, tp = confusion_matrix(targets, predicted, labels=[0, 1]).ravel()
    threshold_rows.append({
        'threshold': float(candidate),
        'sensitivity': tp / (tp + fn) if tp + fn else 0.0,
        'specificity': tn / (tn + fp) if tn + fp else 0.0,
    })
threshold_table = pd.DataFrame(threshold_rows).sort_values('threshold')
ax = threshold_table.plot(
    x='threshold', y=['sensitivity', 'specificity'], figsize=(9, 4),
    color=['#ef4444', '#3b82f6']
)
ax.axvline(threshold, color='black', linestyle='--', label='Selected threshold')
ax.axhline(config['threshold_target_sensitivity'], color='#ef4444', linestyle=':', alpha=0.7)
ax.axhline(config['threshold_min_specificity'], color='#3b82f6', linestyle=':', alpha=0.7)
ax.set(title='Threshold sensitivity-specificity trade-off', ylabel='Metric value')
ax.set_ylim(0, 1.02)
ax.legend()
plt.tight_layout()
threshold_plot = FIGURE_DIR / 'threshold_tradeoff.png'
plt.savefig(threshold_plot, dpi=150)
plt.show()

,value
accuracy,0.721311
precision,0.407407
recall_sensitivity,0.916667
specificity,0.673469
negative_predictive_value,0.970588
macro_f1,0.679642
class_counts,"{'Absent': 98, 'Present': 24}"
roc_auc,0.92517
pr_auc,0.860442
brier_score,0.126363


C:\Users\Andi Alvin\AppData\Local\Temp\ipykernel_13436\445991372.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Andi Alvin\AppData\Local\Temp\ipykernel_13436\445991372.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Optional five-fold grouped cross-validation

Enable the flag near the top to train five additional models. The held-out test split remains untouched.

In [14]:
cross_validation_result = {'executed': False}
if RUN_GROUPED_CROSS_VALIDATION:
    stamp(ROOT, config, 'segments', verify=True)
    started = perf_counter()
    cross_validation_output = cross_validate_cnn(segments, config, ROOT)
    cross_validation_result = {
        'executed': True,
        'elapsed_seconds': perf_counter() - started,
        'fold_count': cross_validation_output['fold_count'],
        'subject_metric_summary': cross_validation_output['subject_metric_summary'],
    }
display(pd.Series({
    'executed': cross_validation_result['executed'],
    'elapsed_seconds': cross_validation_result.get('elapsed_seconds'),
}, name='value').to_frame())

,value
executed,False
elapsed_seconds,None


## Export measured JSON and Markdown reports

In [15]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return value

measured_report = json_ready({
    'schema_version': 1,
    'report_kind': 'spectrogram_cnn_measured_development_report',
    'research_only': True,
    'environment': environment,
    'configuration': {
        'path': str(CONFIG_PATH.relative_to(ROOT)),
        'sha256': hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
        'values': config,
    },
    'automated_tests': test_result,
    'training_run': training_run,
    'dataset': dataset_measurements,
    'leakage_check_passed': True,
    'spectrogram_measurements': spectrogram_measurements,
    'architecture': architecture_measurements,
    'model_artifact_bytes': model_path.stat().st_size,
    'model_artifact_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(),
    'model_metadata': model_metadata,
    'validation': metrics_payload['validation'],
    'threshold_selection': metrics_payload['threshold_selection'],
    'cross_validation': cross_validation_result,
    'holdout': metrics_payload.get('holdout'),
    'figures': [
        str(path.relative_to(ROOT)) for path in [
            class_plot, spectrogram_plot, learning_plot, evaluation_plot, threshold_plot
        ]
    ],
    'warning': 'Development evaluation only; no clinical validity established.',
})
json_path = REPORT_DIR / 'measured_report.json'
json_path.write_text(json.dumps(measured_report, indent=2), encoding='utf-8')

subject = measured_report['validation']['subject']
threshold_info = measured_report['threshold_selection']
markdown = f"""# AurisCore spectrogram-CNN measured report

Generated: {environment['generated_at_utc']}

## Run status

- Full training executed: **{training_run['executed']}**
- Training elapsed: **{training_run.get('elapsed_seconds', 0):.1f} seconds**
- Epochs completed: **{metrics_payload['training_epochs']}**
- Best validation loss: **{metrics_payload['best_validation_loss']:.6f}**
- Automated tests: **{test_result.get('summary', 'not run')}**
- Leakage check: **passed**

## Participant validation metrics

| Metric | Measured value |
|---|---:|
| Sensitivity | {subject['recall_sensitivity']:.4f} |
| Specificity | {subject['specificity']:.4f} |
| Precision | {subject['precision']:.4f} |
| Negative predictive value | {subject['negative_predictive_value']:.4f} |
| Accuracy | {subject['accuracy']:.4f} |
| Macro F1 | {subject['macro_f1']:.4f} |
| ROC-AUC | {subject.get('roc_auc', float('nan')):.4f} |
| PR-AUC | {subject.get('pr_auc', float('nan')):.4f} |
| Brier score | {subject.get('brier_score', float('nan')):.4f} |

Selected threshold: **{threshold:.6f}**. Threshold constraints met: **{threshold_info['constraints_met']}**.

Confusion matrix `[[TN, FP], [FN, TP]]`: `{subject['confusion_matrix']}`.

## Measured implementation

- Model: `{architecture_measurements['model_name']}`
- Parameters: **{architecture_measurements['total_parameters']:,}**
- Spectrogram tensor: **{spectrogram_measurements['tensor_shape']}**
- Feature extraction: **{spectrogram_measurements['milliseconds_per_window']:.3f} ms/window** on this computer
- Saved model size: **{model_path.stat().st_size / 1024:.1f} KiB**
- Model SHA-256: `{measured_report['model_artifact_sha256']}`

## Figures

- `figures/participant_class_distribution.png`
- `figures/spectrogram_examples.png`
- `figures/training_curves.png`
- `figures/participant_evaluation.png`
- `figures/threshold_tradeoff.png`

## Interpretation boundary

These are development-validation measurements. The final holdout was not evaluated. This system remains research-only and is not a diagnosis.
"""
markdown_path = REPORT_DIR / 'measured_report.md'
markdown_path.write_text(markdown, encoding='utf-8')
print(f'JSON report: {json_path}')
print(f'Markdown report: {markdown_path}')
display(pd.DataFrame([subject]))

JSON report: C:\Users\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline\artifacts\notebook_report\measured_report.json
Markdown report: C:\Users\Andi Alvin\Documents\File Kuliah\Semester 7\Despro\Despro-AurisCore-Code\AI-Pipeline\artifacts\notebook_report\measured_report.md


,accuracy,precision,recall_sensitivity,specificity,negative_predictive_value,macro_f1,confusion_matrix,class_counts,roc_auc,pr_auc,brier_score
0,0.721311,0.407407,0.916667,0.673469,0.970588,0.679642,"[[66, 32], [2, 22]]","{'Absent': 98, 'Present': 24}",0.92517,0.860442,0.126363
